In [1]:
import pandas as pd
from transformers import pipeline
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
import numpy as np
import re
import random
random.seed(42)
np.random.seed(42)

### Language style bias: Emotion Bias (Theme Specific)

In [ ]:
import pandas as pd
from transformers import pipeline
from scipy.stats import ttest_ind, f_oneway


# Load pretrained emotion classifier #GoEmotions (Google) 27 fine-grained emotion classes + neutral. we use it in the paper
emotion_clf = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=None  # return probabilities for all labels
)




# ============ 1. Get emotion scores ============
def get_emotion_scores(text):
    preds = emotion_clf(text, truncation=True, max_length=512)[0]
    return {p["label"]: p["score"] for p in preds}

# # Get text emotion 
def get_emotion(df, group_col, text_col="output", model_name="model"):
     # Apply classifier to all texts
    emotion_scores = df[text_col].apply(get_emotion_scores)
    emo_df =  pd.DataFrame(list(emotion_scores))
    #print(group_col)
    emo_df["Text"] = df[text_col]  # Add the original text
    # Step 3: Get the dominant emotion for each text
    dom_emo = [col for col in emo_df.columns if col not in ["Text", group_col, "Model"]]
    emo_df["DominantEmotion"] = emo_df[dom_emo].idxmax(axis=1)
    emo_df["Group"] = df[group_col]  # Add the group either gender or age
    emo_df["Model"] = model_name
    #print(emo_df)
    return emo_df
   
    #scoreemo_dfs_df.to_csv("text_emotion_scores.csv", index=False)

# ============ 2. Core Analysis Function ============
def analyze_emotion_bias(df, group_col, text_col="output", model_name="model"):
    """
    group_col: "gender" (2 groups → t-test) or "age" (4 groups → ANOVA)
    """
    # Apply classifier to all texts
    emotion_scores = df[text_col].apply(get_emotion_scores)
    scores_df = pd.DataFrame(list(emotion_scores))
    scores_df[group_col] = df[group_col].values
    scores_df["Model"] = model_name
    results = []
    emotions = [c for c in scores_df.columns if c not in ["Model", group_col]]
    #print(scores_df[emotions].max(axis = 1)) #print max emotion score
    # scores_df["Max_emo"] = scores_df[emotions].max(axis = 1)
    #print(scores_df)
    for emotion in emotions:
        groups = scores_df.groupby(group_col)[emotion].apply(list)
        #print(groups.index[0], groups.index[1], groups.index[2], groups.index[3])
        if group_col == "gender" and len(groups) == 2:
            # --- Independent t-test ---
            g1, g2 = groups.iloc[0], groups.iloc[1]
            #print(g1, g2)
            t_stat, p_val = ttest_ind(g1, g2, equal_var=False)
            results.append({
                "Model": model_name,
                "GroupType": "Gender",
                "Emotion": emotion,
                f"Mean_{groups.index[0]}": sum(g1)/len(g1),
                f"Mean_{groups.index[1]}": sum(g2)/len(g2),
                "t_stat": t_stat,
                "p_val": p_val
            })

        elif group_col == "age" and len(groups) >= 2:
             # --- paired t-test ---
            #g1, g2 = groups.iloc[3], groups.iloc[2] #young, senior
            g1, g2 = groups.iloc[0], groups.iloc[1] #Early working, Late working
            #print(g1)
            t_stat, p_val = ttest_ind(g1, g2, equal_var=False)
            results.append({
                "Model": model_name,
                "GroupType": "Age",
                "Emotion": emotion,
                # f"Mean_{groups.index[3]}": sum(g1)/len(g1),
                # f"Mean_{groups.index[2]}": sum(g2)/len(g2),
                f"Mean_{groups.index[0]}": sum(g1)/len(g1),
                f"Mean_{groups.index[1]}": sum(g2)/len(g2),
                "t_stat": t_stat,
                "p_val": p_val
            })

    return pd.DataFrame(results)

# ============ 3. Wrapper for All Models ============
def run_bias_analysis(files, group_col):
    all_dfs = []
    
    for model, file in files.items():
        df = pd.read_csv(file)
        #filtered_df = df[df['theme'].isin(['Future Generation'])].reset_index(drop=True) #future gen for gender bias
        #filtered_df = df[df['theme'].isin(['Support climate policy'])].reset_index(drop=True)  #for gender bias
       
        filtered_df = df[df['theme'].isin(['Economy'])].reset_index(drop=True) #economy for age bias
        #filtered_df = df[df['theme'].isin(['Patriotism'])].reset_index(drop=True) ## Patriotism  for age bias
        res = analyze_emotion_bias(filtered_df, group_col=group_col, text_col="output", model_name=model)
        all_dfs.append(res)
        
    return pd.concat(all_dfs, ignore_index=True)

def run_emo_analysis(files, group_col):
    all_dfs_emo = []
    for model, file in files.items(): 
        df = pd.read_csv(file)      
        #filtered_df = df[df['theme'].isin(['Future Generation'])].reset_index(drop=True) #future gen for gender bias
        #filtered_df = df[df['theme'].isin(['Support climate policy'])].reset_index(drop=True)  #for gender bias
        filtered_df = df[df['theme'].isin(['Economy'])].reset_index(drop=True) #economy for age bias
        #filtered_df = df[df['theme'].isin(['Patriotism'])].reset_index(drop=True) ## Patriotism for age bias
        res_emo = get_emotion(filtered_df, group_col=group_col, text_col="output", model_name=model)
        all_dfs_emo.append(res_emo)
    return pd.concat(all_dfs_emo, ignore_index=True)



Device set to use cpu


In [ ]:


files = {
    "gpt-4o": "/Data/gpt4o_climate.csv",
    "llama-3.3": "/Data/llama3.3_climate.csv",
    "mistral-2.1": "/Data/mistral_large2.1_climate.csv"
}


# ### Gender-based emotion bias
gender_bias = run_bias_analysis(files, group_col="gender")
gender_emo = run_emo_analysis(files, group_col="gender")



# # Age-based emotion bias
age_bias = run_bias_analysis(files, group_col="age")
age_emo = run_emo_analysis(files, group_col="age")




In [ ]:
#gender_emo
#age_emo = age_emo[['Text', 'DominantEmotion', 'Group', 'Model']]
age_emo.columns